In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.readers import (
    InputExample,
) # Added this import
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report
from sklearn.metrics.pairwise import cosine_similarity

import json, warnings
warnings.filterwarnings('ignore')

print('Libraries Imported!!')

In [ ]:
df = pd.read_csv('dataset/cleaned_resumeJD_pairs.csv', engine='python', on_bad_lines='skip')
print(f"Loaded: {len(df)} pairs")
print(f"\nLabel Distribution: ")
print(df['label'].value_counts())
print(f'\nScore range: {df["match_score"].min():.2f} - {df["match_score"].max():.2f}')
df.head(3)

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f'Train:  {len(train_df)} pairs')
print(f'Val:    {len(val_df)} pairs')
print(f'Test:   {len(test_df)} pairs')
print()

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name} Label Distribution: {split["label"].value_counts()}')

In [ ]:
train_examples=[
    InputExample(texts=[row['resume_text'], row['job_description_text']], label=float(row['match_score']))
    for _, row in train_df.iterrows()
]

val_examples=[
    InputExample(texts=[row['resume_text'], row['job_description_text']], label=float(row['match_score']))
    for _, row in val_df.iterrows()
]

print(f"Train Examples: {len(train_examples)}")
print(f"Val Examples: {len(val_examples)}")
print("\nSample Input Examples")
print(f" text1(resume) : {train_examples[0].texts[0][:80]}...")
print(f" text2(JD) :     {val_examples[0].texts[1][:80]}...")
print(f" Label : {train_examples[0].label}")

In [ ]:
print("Loading Base BERT model...")
base_model = SentenceTransformer('all-mpnet-base-v2')
print("Model Loaded!!")

In [ ]:
print("Evaluating base model on test set...")

base_preds = []
for _, row in test_df.iterrows():
  emb1 = base_model.encode(row['resume_text'])
  emb2 = base_model.encode(row['job_description_text'])
  sim = cosine_similarity([emb1], [emb2])[0][0]
  base_preds.append(float(sim))

base_mae = mean_absolute_error(test_df['match_score'], base_preds)
base_rmse = np.sqrt(mean_squared_error(test_df['match_score'], base_preds))

print("Base Model - Test Set Performance")
print(f"Base Model MAE: {base_mae:.4f}")
print(f"Base Model RMSE: {base_rmse:.4f}")
print()
print("Goal: fine-tuning should reduce MAE below this number.")

In [ ]:
from sentence_transformers import losses
model = SentenceTransformer('all-mpnet-base-v2')
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples, name='ats-val'
)

print("Training Setup ready...")
print(" Batch Size:  16")
print(f" Train pairs: {len(train_examples)}")
print(f" Steps/Epoch: {len(train_dataloader)}")

In [ ]:
total_steps = len(train_dataloader) * 10
warmup_steps = int(total_steps * 0.1)

print(f"Total Steps: {total_steps}")
print(f"Warmup Steps: {warmup_steps}")
print("\nStarting fine-tuning...\n")


model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=10,
    evaluation_steps=len(train_dataloader),
    warmup_steps=warmup_steps,
    output_path='models/finetuned-bert',
    save_best_model=True,
    use_amp=True,
    show_progress_bar=True
)

print("FINE TUNING COMPLETE!!")

In [ ]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

# 1. Increase training intensity
# Try a slightly lower learning rate and more focus on hard negatives if available
model_optimized = SentenceTransformer("all-mpnet-base-v2")
train_dataloader_opt = DataLoader(
    train_examples, shuffle=True, batch_size=32
)  # Increased batch size
train_loss_opt = losses.CosineSimilarityLoss(model=model_optimized)

# 2. Re-run fit with adjusted parameters
model_optimized.fit(
    train_objectives=[(train_dataloader_opt, train_loss_opt)],
    evaluator=evaluator,
    epochs=15,  # Increased epochs
    warmup_steps=int(len(train_dataloader_opt) * 15 * 0.1),
    output_path="models/finetuned-bert-v2",
    optimizer_params={"lr": 2e-5},  # Specific learning rate
    save_best_model=True,
    use_amp=True,
)

print("Optimized Fine-Tuning Complete!")

In [ ]:
finetuned_model = SentenceTransformer('models/finetuned-bert')
print('Fine Tuned Model Loaded')

In [ ]:
ft_preds = []
for _, row in test_df.iterrows():
  emb1 = finetuned_model.encode(row['resume_text'])
  emb2 = finetuned_model.encode(row['job_description_text'])
  sim = cosine_similarity([emb1], [emb2])[0][0]
  ft_preds.append(float(sim))

# Fixed: using ft_preds instead of base_preds
ft_mae = mean_absolute_error(test_df['match_score'], ft_preds)
ft_rmse = np.sqrt(mean_squared_error(test_df['match_score'], ft_preds))

print("Fine-Tuned Model - Test Set Performance")
print(f"FT Model MAE: {ft_mae:.4f}")
print(f"FT Model RMSE: {ft_rmse:.4f}")

In [ ]:
print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"Base Model MAE: {base_mae:.4f}")
print(f"Fine Tuned MAE: {ft_mae:.4f}")
print(f"Improvement: {base_mae - ft_mae:.2f}%")
print("=" * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = test_df['label'].map({'no fit':'red', 'potential fit': 'orange', 'good fit': 'green'})

for ax, preds, title, mae in [(axes[0], base_preds, 'Base Model', base_mae),
    (axes[1], ft_preds, 'Fine-Tuned Model', ft_mae)]:

    ax.scatter(test_df['match_score'], preds, color=colors)
    ax.plot([0, 1], [0, 1], 'k--', label="Perfect")
    ax.set_xlabel("Match Score")
    ax.set_ylabel("Predicted Match Score")
    ax.set_title(f"{title} (MAE: {mae:.4f})")
    ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch
# Fixed: changed 'lower_center' to 'lower center'
fig.legend(handles=[
    Patch(color='green', label='Good Fit'),
    Patch(color='orange', label='Potential Fit'),
    Patch(color='red', label='No Fit')
], loc='lower center', ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
metadata = {
    'base_model':     'all-mpnet-base-v2',
    'dataset':        'merged_dataset_clean.csv',
    'total_pairs':    len(df),
    'train_pairs':    len(train_df),
    'val_pairs':      len(val_df),
    'test_pairs':     len(test_df),
    'epochs':         10,
    'batch_size':     16,
    'base_mae':       round(float(base_mae), 4),
    'ft_mae':         round(float(ft_mae), 4),
    'improvement_pct':    round((base_mae - ft_mae) / base_mae)
}

with open('models/finetuned-bert/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: models/finetuned-bert/')
print(json.dumps(metadata, indent=2))

In [ ]:
import os

# List all directories in the current path
current_path = os.getcwd()
dirs = [d for d in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, d))]

print(f"Current Directory: {current_path}")
print("Folders found:", dirs)

if 'models' in dirs:
    print("\n✅ The 'models' folder exists! Click the Refresh button in the file sidebar to see it.")
else:
    print("\n❌ The 'models' folder was not found in the current directory.")

### Permanent Storage: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Copy the Model Folder to Drive
This will create a folder named `finetuned_models_backup` in your main My Drive folder.

In [ ]:
!mkdir -p "/content/drive/My Drive/finetuned_models_backup"
!cp -r models/finetuned-bert "/content/drive/My Drive/finetuned_models_backup/"
print("Copy complete! Check your Google Drive for the 'finetuned_models_backup' folder.")